In [1]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
#import scipy.stats as stats
#from scipy import signal
from netCDF4 import Dataset
 
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap

import rasterio
from rasterio.transform import from_origin

import os

/home/anas.oubaha/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
!free -g

              total        used        free      shared  buff/cache   available
Mem:            376          59         143           1         172         312
Swap:             0           0           0


**Read NC data**

In [2]:
# Load dataset
dw = xr.open_dataset('data/850&200PH_Data_80N20S50W80E.nc', engine="netcdf4")  # Load NetCDF file
dw = dw[['r']] #keep only the variable r
#dw = dw.sel(latitude=slice(45, 15))
#dw = dw.sel(valid_time=slice('1991-01-01', None))

dwl = dw.where(dw.pressure_level.isin([850]), drop=True)
dwl = dwl.squeeze()
dwh = dw.where(dw.pressure_level.isin([200]), drop=True)
dwh = dwh.squeeze()

In [3]:
dwl

<xarray.Dataset>
Dimensions:         (valid_time: 408, latitude: 401, longitude: 521)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 1991-01-01 ... 2024-12-01
    pressure_level  float64 850.0
  * latitude        (latitude) float64 80.0 79.75 79.5 ... -19.5 -19.75 -20.0
  * longitude       (longitude) float64 -50.0 -49.75 -49.5 ... 79.5 79.75 80.0
Data variables:
    r               (valid_time, latitude, longitude) float32 91.44 ... 72.86
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

**Seasonal Relative Humidity anomalies**

In [4]:
# Convert 'date' from integer to datetime format
dwl = dwl.assign_coords(date=pd.to_datetime(dwl['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
df = dwl['r'].to_dataframe().reset_index()

# Extract year and month
df['year'] = df['valid_time'].dt.year
df['month'] = df['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = df['month'].isin([12, 1, 2])
spring = df['month'].isin([3, 4, 5])
summer = df['month'].isin([6, 7, 8])
autumn = df['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
df['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
df['season_year'] = df['year']
df.loc[df['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_r_df = df.groupby(['latitude', 'longitude', 'season_year', 'season'])['r'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years)**
seasonal_climatology_df = seasonal_r_df.groupby(['latitude', 'longitude', 'season'])['r'].mean().reset_index()
seasonal_climatology_df.rename(columns={'r': 'r_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedf = pd.merge(seasonal_r_df, seasonal_climatology_df, on=['latitude', 'longitude', 'season'], how="left")
# **STEP 4: Compute Seasonal Anomalies**
mergedf['r_anomaly'] = mergedf['r'] - mergedf['r_climatology']

mergedf['date'] = (
    mergedf['season'].astype(str) 
    + ' ' 
    + mergedf['season_year'].astype(str)
)
########################################################################################################
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a unique season key
#mergedf['season_key'] = mergedf['season'] + '_' + mergedf['season_year'].astype(str)

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedf.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='r_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
#t_df = pd.DataFrame([t_vals], columns=pivot.columns).T.reset_index()
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
#t_df.columns = ['latitude', 'longitude', 'season', 't_value']
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge t/p values back
#significance_df = pd.merge(t_df, p_df, on=['latitude', 'longitude', 'season'])

# Merge with original mergedf
mergedf = pd.merge(mergedf, p_df, on=['latitude', 'longitude', 'season'], how='left')
#########################################################################################
# Convert back to xarray
anomalies = mergedf.set_index(['date', 'latitude', 'longitude'])['r_anomaly'].to_xarray()
pval = mergedf.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
anomalies

<xarray.DataArray 'r_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[  5.4244995 ,   5.722786  ,   6.0407944 , ...,   0.9873123 ,
           0.9509125 ,   0.957901  ],
        [  5.3826065 ,   5.7963333 ,   6.2034607 , ...,   1.1480103 ,
           1.0388947 ,   1.001915  ],
        [  5.4952545 ,   6.0031204 ,   6.432251  , ...,   1.0596161 ,
           0.7996292 ,   0.66197205],
        ...,
        [ -0.7599716 ,  -0.826004  ,  -0.8920746 , ...,   1.4661255 ,
           1.5870743 ,   1.6899719 ],
        [ -0.1357956 ,  -0.23236847,  -0.33641815, ...,   1.3815613 ,
           1.5397491 ,   1.6837769 ],
        [  0.4919281 ,   0.39026642,   0.2854004 , ...,   1.233223  ,
           1.3873291 ,   1.5358582 ]],

       [[  4.1521072 ,   3.9442139 ,   3.7091599 , ...,   4.8438263 ,
           4.9067154 ,   4.8765945 ],
        [  3.9334564 ,   3.7648315 ,   3.6459122 , ...,   4.8681335 ,
           4.9195023 ,   4.9427414 ],
        [  3.9181747 ,   3.8641968 ,   3.8747025 , ...,   4.927849  ,
           4.9705963 ,   5.020439  ],
...
        [ -1.7809753 ,  -1.7618484 ,  -1.7359924 , ...,   0.8826904 ,
           0.83815   ,   0.80168915],
        [ -1.6872635 ,  -1.6372299 ,  -1.5865784 , ...,   0.39383698,
           0.3790512 ,   0.37844086],
        [ -1.6051102 ,  -1.5590591 ,  -1.5166321 , ...,   0.347229  ,
           0.35331726,   0.3703003 ]],

       [[ -7.1158524 ,  -7.1416435 ,  -7.1268005 , ...,   7.369507  ,
           7.347336  ,   7.435959  ],
        [ -6.8925247 ,  -6.8803062 ,  -6.8907967 , ...,   7.4179077 ,
           7.453697  ,   7.548958  ],
        [ -6.526428  ,  -6.524433  ,  -6.5375557 , ...,   7.349373  ,
           7.44738   ,   7.5130386 ],
        ...,
        [ -0.23692322,  -0.26857758,  -0.32476044, ...,   1.0816803 ,
           0.9570694 ,   0.8483429 ],
        [ -0.17381287,  -0.2012558 ,  -0.24240112, ...,   1.1162643 ,
           0.9797363 ,   0.85868835],
        [ -0.03697205,  -0.04170227,  -0.05591583, ...,   1.2239532 ,
           1.0764008 ,   0.94169617]]], dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [5]:
import os
def export_to_geotiff(path, u_data, v_data, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=2,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2)  # Band 2 - v
        
# Export regression slope map
output_dir = 'RH850'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(anomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, anomalies[d], pval[d], anomalies.latitude.values, anomalies.longitude.values, 'RH850 Seasonal Anomalies of '+str(anomalies['date'][d].values))


In [6]:
# Convert 'date' from integer to datetime format
dwh = dwh.assign_coords(date=pd.to_datetime(dwh['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
df = dwh['r'].to_dataframe().reset_index()

# Extract year and month
df['year'] = df['valid_time'].dt.year
df['month'] = df['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = df['month'].isin([12, 1, 2])
spring = df['month'].isin([3, 4, 5])
summer = df['month'].isin([6, 7, 8])
autumn = df['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
df['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
df['season_year'] = df['year']
df.loc[df['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_r_df = df.groupby(['latitude', 'longitude', 'season_year', 'season'])['r'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years)**
seasonal_climatology_df = seasonal_r_df.groupby(['latitude', 'longitude', 'season'])['r'].mean().reset_index()
seasonal_climatology_df.rename(columns={'r': 'r_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedf = pd.merge(seasonal_r_df, seasonal_climatology_df, on=['latitude', 'longitude', 'season'], how="left")
# **STEP 4: Compute Seasonal Anomalies**
mergedf['r_anomaly'] = mergedf['r'] - mergedf['r_climatology']

mergedf['date'] = (
    mergedf['season'].astype(str) 
    + ' ' 
    + mergedf['season_year'].astype(str)
)
########################################################################################################
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a unique season key
#mergedf['season_key'] = mergedf['season'] + '_' + mergedf['season_year'].astype(str)

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedf.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='r_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
#t_df = pd.DataFrame([t_vals], columns=pivot.columns).T.reset_index()
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
#t_df.columns = ['latitude', 'longitude', 'season', 't_value']
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge t/p values back
#significance_df = pd.merge(t_df, p_df, on=['latitude', 'longitude', 'season'])

# Merge with original mergedf
mergedf = pd.merge(mergedf, p_df, on=['latitude', 'longitude', 'season'], how='left')
#########################################################################################
# Convert back to xarray
anomalies = mergedf.set_index(['date', 'latitude', 'longitude'])['r_anomaly'].to_xarray()
pval = mergedf.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
anomalies

<xarray.DataArray 'r_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[ -4.8060074 ,  -4.566353  ,  -4.659378  , ...,   1.7959595 ,
           2.1066742 ,   2.3781052 ],
        [ -4.506172  ,  -4.3999786 ,  -4.510994  , ...,   1.7301483 ,
           2.0458145 ,   2.3988266 ],
        [ -4.1719666 ,  -4.2361298 ,  -4.2208405 , ...,   1.9195633 ,
           2.2605057 ,   2.6219864 ],
        ...,
        [ -7.9513626 ,  -7.9639378 ,  -7.976288  , ...,  -5.4407024 ,
          -5.486744  ,  -5.532593  ],
        [ -8.172325  ,  -8.185215  ,  -8.197456  , ...,  -5.2433834 ,
          -5.2929344 ,  -5.341337  ],
        [ -8.378855  ,  -8.392508  ,  -8.405231  , ...,  -5.141371  ,
          -5.19141   ,  -5.241745  ]],

       [[ -1.7450104 ,  -1.8774261 ,  -1.8926544 , ...,  16.201038  ,
          15.957066  ,  15.719379  ],
        [ -1.3592377 ,  -1.3122864 ,  -1.3230438 , ...,  15.74102   ,
          15.596073  ,  15.344269  ],
        [ -0.90719604,  -0.8125    ,  -0.88575745, ...,  15.197037  ,
          15.060764  ,  14.844772  ],
...
        [  5.176422  ,   5.197752  ,   5.2194853 , ...,   6.6142664 ,
           6.5934315 ,   6.5717754 ],
        [  4.743038  ,   4.7642736 ,   4.7861214 , ...,   6.687834  ,
           6.664549  ,   6.6414375 ],
        [  4.3409624 ,   4.3644953 ,   4.3889866 , ...,   6.7818527 ,
           6.7521152 ,   6.720482  ]],

       [[ -3.9010162 ,  -3.741932  ,  -3.6114388 , ...,   7.6244125 ,
           7.3318253 ,   7.006077  ],
        [ -3.6566887 ,  -3.478489  ,  -3.2617874 , ...,   7.1842155 ,
           6.829357  ,   6.4843025 ],
        [ -3.5973015 ,  -3.417286  ,  -3.1770515 , ...,   6.5805244 ,
           6.272064  ,   6.0358315 ],
        ...,
        [  0.3459034 ,   0.306036  ,   0.26526928, ...,  -1.0534916 ,
          -1.04568   ,  -1.0373878 ],
        [  0.25301552,   0.21240139,   0.17109871, ...,  -1.1726332 ,
          -1.1640158 ,  -1.1558771 ],
        [  0.11981869,   0.08345699,   0.04675007, ...,  -1.2849941 ,
          -1.2763205 ,  -1.2669382 ]]], dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [7]:
import os
def export_to_geotiff(path, u_data, v_data, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=2,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2)  # Band 2 - v
        
# Export regression slope map
output_dir = 'RH200'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(anomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, anomalies[d], pval[d], anomalies.latitude.values, anomalies.longitude.values, 'RH200 Seasonal Anomalies of '+str(anomalies['date'][d].values))
